# Probability and Statistics Capstone — Fraud Alert Evaluation

## Objective

Analyze a rare-event synthetic transaction dataset, choose an alert threshold on training data, and evaluate conditional probabilities and classification outcomes on held-out data.

**Prerequisites:** Probability Notebooks 01–03 and applied statistics topics on summaries, outliers, and confidence intervals. This capstone uses no advanced decomposition or deep learning.

**Working rule:** Complete the starter cells yourself. Verification cells are deliberately placed after each task and will fail until that task is solved. Keep the assertions; catch only failures that the task explicitly asks you to provoke. Use NumPy plus Python's standard library only. After finishing, restart the kernel and run all cells in order.

**Why this matters:** A useful fraud workflow must separate prevalence, alert sensitivity, alert precision, and threshold selection. Rare events make a high detection rate compatible with many false alerts.


## Problem and synthetic dataset

A review team can inspect only a limited number of transactions. Choose a score threshold using the training subset, then estimate how the rule performs on untouched test data. The generator below creates realistic **synthetic** variation in amount, recent activity, and account age, with rare fraud labels. It is supplied as data setup, not a model implementation task. No private records or external downloads are needed. Do not use test labels to choose the threshold.


In [ ]:
import numpy as np

rng = np.random.default_rng(2026)
n = 2400
amount = rng.lognormal(mean=4.5, sigma=.7, size=n)
recent_count = rng.poisson(lam=2, size=n).astype(float)
account_age_days = rng.exponential(scale=365, size=n)
latent = -4.5 + .007 * amount + .35 * recent_count - .001 * account_age_days
fraud_probability = 1 / (1 + np.exp(-latent))
fraud = rng.binomial(1, fraud_probability).astype(bool)
risk_score = (.007 * amount + .35 * recent_count - .001 * account_age_days
              + rng.normal(0, .3, size=n))
X = np.column_stack((amount, recent_count, account_age_days))
cut = int(.7 * n)
X_train, X_test = X[:cut], X[cut:]
y_train, y_test = fraud[:cut], fraud[cut:]
score_train, score_test = risk_score[:cut], risk_score[cut:]
assert X_train.shape == (1680, 3) and X_test.shape == (720, 3)
assert 0 < y_train.sum() < len(y_train) and 0 < y_test.sum() < len(y_test)


## Task 1 — Audit prevalence and feature distributions

### Goal

Describe the rare event and check dataset shape before evaluating alerts.

### Required work

1. Implement `summarize_feature(values)` returning mean, median, sample standard deviation, and 90th percentile.
2. Summarize amount separately for training fraud and legitimate transactions; report prevalence in both splits.
3. Explain what the mean/median difference suggests about the amount distribution.

### Expected output

Two four-number summaries and two prevalence estimates in `(0,1)`.

### Acceptance criteria

- [ ] Only training data is used for the class summaries.
- [ ] Statistics match NumPy reference calculations.
- [ ] Empty input raises `ValueError`.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def summarize_feature(values):
    # TODO: validate 1D nonempty data; return four summary numbers.
    raise NotImplementedError

# TODO: calculate prevalence_train and prevalence_test.
prevalence_train = prevalence_test = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
summary = summarize_feature(X_train[:, 0])
assert len(summary) == 4
assert np.allclose(summary, (np.mean(X_train[:, 0]), np.median(X_train[:, 0]),
                             np.std(X_train[:, 0], ddof=1), np.quantile(X_train[:, 0], .9)))
assert prevalence_train is not None and np.isclose(prevalence_train, y_train.mean())
assert prevalence_test is not None and np.isclose(prevalence_test, y_test.mean())
try:
    summarize_feature([])
except ValueError:
    pass
else:
    raise AssertionError('Empty summary was accepted')


## Task 2 — Covariance and association checks

### Goal

Inspect feature relationships before deciding what a risk score means.

### Required work

1. Compute the training feature covariance matrix and correlation matrix with observations as rows.
2. Calculate the correlation of each feature with the binary fraud label on training data only.
3. Explain why correlation does not establish causation and why labels generated after an event would cause leakage.

### Expected output

Two `(3,3)` symmetric matrices and three finite feature-label correlations.

### Acceptance criteria

- [ ] Shapes and diagonal properties are checked.
- [ ] Test labels are not used.
- [ ] Interpretation distinguishes association from causation.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: assign feature_cov, feature_corr, label_corr.
feature_cov = feature_corr = label_corr = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert feature_cov is not None and np.asarray(feature_cov).shape == (3, 3)
assert feature_corr is not None and np.asarray(feature_corr).shape == (3, 3)
assert label_corr is not None and np.asarray(label_corr).shape == (3,)
assert np.allclose(feature_cov, np.cov(X_train, rowvar=False, ddof=1))
assert np.allclose(feature_corr, np.corrcoef(X_train, rowvar=False))
assert np.allclose(feature_cov, feature_cov.T)
assert np.allclose(feature_corr, feature_corr.T)
assert np.allclose(np.diag(feature_corr), 1)
assert np.all(np.isfinite(label_corr)) and np.all(np.abs(label_corr) <= 1 + 1e-12)


## Task 3 — Alert probabilities and Bayes check

### Goal

Calculate what an alert means at a provisional threshold.

### Required work

1. At provisional threshold `1.5`, create training alert flags and a 2-by-2 count table with layout `[[TP, FN], [FP, TN]]`.
2. From counts, calculate `P(alert | fraud)`, `P(alert | legitimate)`, and `P(fraud | alert)`.
3. Independently use Bayes' rule with the training prevalence and conditional rates; compare results.

### Expected output

Four nonnegative counts totaling 1680 and a Bayes posterior matching direct alert precision.

### Acceptance criteria

- [ ] No denominator is zero without a documented policy.
- [ ] Direct and Bayes posteriors agree numerically.
- [ ] All calculations use the training split.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: create provisional_alerts, provisional_counts, provisional_rates,
# where rates are (sensitivity, false_alert_rate, precision, bayes_precision).
provisional_alerts = provisional_counts = provisional_rates = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert provisional_alerts is not None and len(provisional_alerts) == len(y_train)
assert np.array_equal(provisional_alerts, score_train >= 1.5)
assert provisional_counts is not None and np.asarray(provisional_counts).shape == (2, 2)
assert np.asarray(provisional_counts).sum() == len(y_train)
expected_counts = np.array([[np.sum(provisional_alerts & y_train),
                             np.sum(~provisional_alerts & y_train)],
                            [np.sum(provisional_alerts & ~y_train),
                             np.sum(~provisional_alerts & ~y_train)]])
assert np.array_equal(provisional_counts, expected_counts)
assert provisional_rates is not None and len(provisional_rates) == 4
assert np.isclose(provisional_rates[2], provisional_rates[3])
assert all(0 <= z <= 1 for z in provisional_rates)


## Task 4 — Build reusable evaluation metrics

### Goal

Keep denominators and undefined cases explicit.

### Required work

1. Implement `confusion_counts(truth, alert)` returning `(tp, fp, tn, fn)`.
2. Implement `precision_recall(truth, alert)` using those counts and state a policy for zero predicted alerts.
3. Verify a small hand-checkable example before using the full dataset.

### Expected output

For truth `[1,0,1,0]` and alert `[1,1,0,0]`, counts `(1,1,1,1)` and metrics `(0.5,0.5)`.

### Acceptance criteria

- [ ] Mismatched lengths raise `ValueError`.
- [ ] Zero-alert policy is tested.
- [ ] The same functions can evaluate train and test.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def confusion_counts(truth, alert):
    # TODO: validate equal-length 1D binary arrays; return tp, fp, tn, fn.
    raise NotImplementedError

def precision_recall(truth, alert):
    # TODO: use confusion_counts; define zero-alert precision as 0.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
t = np.array([1, 0, 1, 0])
a = np.array([1, 1, 0, 0])
assert tuple(confusion_counts(t, a)) == (1, 1, 1, 1)
assert np.allclose(precision_recall(t, a), (.5, .5))
assert precision_recall(t, np.zeros(4, dtype=int))[0] == 0
try:
    confusion_counts(t, a[:-1])
except ValueError:
    pass
else:
    raise AssertionError('Mismatched lengths were accepted')


## Task 5 — Choose a threshold without test leakage

### Goal

Trade alert precision against recall on the training split only.

### Required work

1. Create candidate thresholds from 21 evenly spaced training-score quantiles.
2. For each threshold, calculate training precision and recall; keep candidates with recall at least `0.70`.
3. Select the candidate with highest training precision; break ties by higher threshold. Record review load (alert fraction).

### Expected output

One threshold from the candidate list and its three training metrics.

### Acceptance criteria

- [ ] Threshold selection uses no test labels or scores.
- [ ] Selected recall is at least `.70`.
- [ ] Selection is deterministic.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: create thresholds, selected_threshold, train_precision,
# train_recall, and train_review_fraction.
thresholds = selected_threshold = train_precision = train_recall = train_review_fraction = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert thresholds is not None and np.asarray(thresholds).shape == (21,)
assert np.allclose(thresholds, np.quantile(score_train, np.linspace(0, 1, 21)))
assert selected_threshold is not None and np.any(np.isclose(selected_threshold, thresholds))
assert train_precision is not None and 0 <= train_precision <= 1
assert train_recall is not None and train_recall >= .70
assert train_review_fraction is not None and 0 <= train_review_fraction <= 1
assert np.isclose(train_review_fraction, np.mean(score_train >= selected_threshold))


## Task 6 — Held-out evaluation and uncertainty

### Goal

Report performance once on test data and distinguish estimates from guarantees.

### Required work

1. Apply the selected threshold to `score_test` without changing it; report test confusion counts, precision, recall, and alert fraction.
2. Compute a simple approximate 95% confidence interval for the **test fraud prevalence** using a sample proportion and standard error; discuss its limitations for rare events.
3. Compare train and test results, describe the review workload, and state why this synthetic rule is not production-ready.

### Expected output

Held-out metrics, a two-number interval inside `[0,1]`, and a concise decision memo.

### Acceptance criteria

- [ ] Threshold is unchanged on test.
- [ ] Confusion counts total 720.
- [ ] Interval contains the observed test prevalence.
- [ ] Limitations mention synthetic labels and sampling uncertainty.

**Hint:** Hint: For an approximate proportion interval, use p ± 1.96 × sqrt(p(1-p)/n), then clip to [0,1]. Discuss why approximation may be weak when positives are scarce.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: create test_alerts, test_counts, test_precision, test_recall,
# test_review_fraction, prevalence_interval.
test_alerts = test_counts = test_precision = test_recall = test_review_fraction = prevalence_interval = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert test_alerts is not None and np.array_equal(test_alerts, score_test >= selected_threshold)
assert test_counts is not None and sum(test_counts) == len(y_test)
assert test_precision is not None and 0 <= test_precision <= 1
assert test_recall is not None and 0 <= test_recall <= 1
assert test_review_fraction is not None and np.isclose(test_review_fraction, np.mean(test_alerts))
assert prevalence_interval is not None and len(prevalence_interval) == 2
lo, hi = prevalence_interval
assert 0 <= lo <= prevalence_test <= hi <= 1
half_width = 1.96 * np.sqrt(prevalence_test * (1 - prevalence_test) / len(y_test))
assert np.allclose(prevalence_interval,
                   (max(0, prevalence_test - half_width),
                    min(1, prevalence_test + half_width)))


## Completion check

- [ ] All task cells are implemented and all provided assertions pass.
- [ ] Expected exceptions are caught inside the relevant test cells.
- [ ] **Restart Kernel and Run All** succeeds in notebook order.
- [ ] Each task has a short written explanation in your own words.
- [ ] No real private data, credentials, or large generated files were added.

**Conceptual exit questions**

1. Why can alert precision differ from alert sensitivity?
2. Why must threshold selection use training data only?
3. How does prevalence influence the value of an alert?
4. What would you need before trusting this rule with real transactions?

**Final deliverable:** A rerunnable notebook with synthetic data audit, association checks, Bayes verification, threshold choice, held-out precision/recall/review load, an approximate prevalence interval, and a short decision memo.
